# 🔄 ทดสอบ Tokenizer ใหม่

ทดสอบ tokenizer แบบใหม่ที่ปรับปรุงให้:
1. ใช้ attacut (ถ้าติดตั้ง) หรือ fallback ไป newmm
2. กรองตัวเลขและคำสั้นออก
3. ทำความสะอาดข้อความดีขึ้น

In [1]:
import joblib

X_under = joblib.load('../InitData/result/X_under.pkl')


In [2]:
import re
import pandas as pd
from tqdm import tqdm
from pythainlp import word_tokenize
from pythainlp.corpus.common import thai_stopwords
from sklearn.feature_extraction import text

# Prepare stopwords set (reuse existing logic)
thai_stopwords_set = set(thai_stopwords())
english_stopwords_set = set(text.ENGLISH_STOP_WORDS)
thai_stopwords_set.update([
    'ของ', 'ใน', 'ที่', 'จาก', 'ไป', 'ด้วย', 'กับ', 'และ', 'หรือ', 'แต่', 'เพราะ',
    'ถ้า', 'หาก', 'เขา', 'คุณ', 'ฉัน', 'เรา', 'ทุกคน', 'นี้', 'นั้น', 'คือ', 'การ',
    'ได้', 'มี', 'เป็น', 'ซึ่ง', 'ว่า', 'ก็', 'โดย', 'เช่น', 'เพื่อ'
])
all_stopwords = thai_stopwords_set.union(english_stopwords_set)

# Improved clean_text: keep only Thai/Eng/digits and normalize whitespace
def clean_text_improved(text_input: str) -> str:
    s = str(text_input)
    # unify unicode spaces and remove control chars
    s = s.replace('\u00A0', ' ').replace('\u200b', ' ').strip()
    s = s.lower().strip()
    # remove URLs / mentions / hashtags
    s = re.sub(r'http\S+|www\S+|https\S+', ' ', s)
    s = re.sub(r'@\w+|#\w+', ' ', s)
    # keep Thai chars (ก-๙), latin, digits and spaces; replace others with space
    s = re.sub(r'[^ก-๙a-zA-Z0-9\s]', ' ', s)
    # collapse multiple spaces
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# Improved hybrid tokenizer: try attacut (if installed) for Thai, else fallback to pythainlp newmm
def hybrid_tokenizer_improved(text_input: str, tokenizer_preference: str = 'auto'):
    s = clean_text_improved(text_input)
    # split into contiguous Thai / English / digits segments
    segments = re.findall(r'[a-zA-Z]+|[ก-๙]+|\d+', s)

    # try to import attacut (optional, gives often better Thai segmentation)
    attacut_tokenize = None
    if tokenizer_preference in ('auto', 'attacut'):
        try:
            import attacut
            attacut_tokenize = attacut.tokenize
        except Exception:
            attacut_tokenize = None

    tokens = []
    for seg in segments:
        if re.match(r'[ก-๙]+', seg):
            # Thai segmentation
            if tokenizer_preference == 'attacut' and attacut_tokenize is None:
                toks = word_tokenize(seg, engine='newmm')
            else:
                if attacut_tokenize is not None:
                    toks = attacut_tokenize(seg)
                else:
                    toks = word_tokenize(seg, engine='newmm')
            tokens.extend(toks)
        else:
            # english or numeric tokens
            tokens.append(seg.lower())

    # filter tokens: remove stopwords, single-char, and pure digits
    filtered = []
    for w in tokens:
        if not w:
            continue
        if w in all_stopwords:
            continue
        if len(w) <= 1:
            continue
        if re.fullmatch(r'\d+', w):
            continue
        filtered.append(w)

    return filtered

# Old/simple tokenizer (for comparison) - original behaviour simplified here
def hybrid_tokenizer_old(text_input: str):
    s = clean_text_improved(text_input)
    segments = re.findall(r'[a-zA-Z]+|[ก-๙]+|\d+', s)
    tokens = []
    for seg in segments:
        if re.match(r'[ก-๙]+', seg):
            tokens.extend(word_tokenize(seg, engine='newmm'))
        else:
            tokens.append(seg.lower())
    return [w for w in tokens if w not in all_stopwords and len(w) > 1]

In [3]:
# Quick test / comparison on sample sentences
sample_texts = [
    "นายกรัฐมนตรี กล่าวว่าการเรียนรู้ด้วยตนเองเป็นวิธีที่ดีที่สุดในการพัฒนาทักษะ และต้องระวังเว็บไซต์หลอกลวง เช่น http://fake.example.com",
    "ปชช.โวย! ราคาหมูแพงทะลุ 300 บาท/กก. #Saveหมูแพง @PriceControl",
    "สธ.เผยยอดผู้ติดเชื้อลดลง 50% เทียบกับสัปดาห์ที่แล้ว ขณะที่อัตราการฉีดวัคซีนเพิ่มขึ้น"
]

print("=== เปรียบเทียบ tokenizer เก่า vs ใหม่ ===\n")
for i, text in enumerate(sample_texts, 1):
    print(f"\n🔸 ตัวอย่างที่ {i}:")
    print(f"ข้อความ: {text}")
    
    print("\n-- tokenizer เก่า --")
    print(hybrid_tokenizer_old(text))
    
    print("\n-- tokenizer ใหม่ (auto) --")
    print(hybrid_tokenizer_improved(text, tokenizer_preference='auto'))
    
    # ถ้าติดตั้ง attacut แล้ว ลองเปรียบเทียบผลลัพธ์
    try:
        import attacut
        print("\n-- tokenizer ใหม่ (attacut) --")
        print(hybrid_tokenizer_improved(text, tokenizer_preference='attacut'))
    except ImportError:
        print("\nℹ️ ติดตั้ง attacut เพื่อเปรียบเทียบผลลัพธ์เพิ่มเติม: pip install attacut")

=== เปรียบเทียบ tokenizer เก่า vs ใหม่ ===


🔸 ตัวอย่างที่ 1:
ข้อความ: นายกรัฐมนตรี กล่าวว่าการเรียนรู้ด้วยตนเองเป็นวิธีที่ดีที่สุดในการพัฒนาทักษะ และต้องระวังเว็บไซต์หลอกลวง เช่น http://fake.example.com

-- tokenizer เก่า --
['นายกรัฐมนตรี', 'กล่าวว่า', 'การเรียนรู้', 'วิธี', 'ดี', 'การพัฒนา', 'ทักษะ', 'ระวัง', 'เว็บไซต์', 'หลอกลวง']

-- tokenizer ใหม่ (auto) --
['นายก', 'รัฐมนตรี', 'เรียนรู้', 'วิธี', 'ดี', 'พัฒนา', 'ทักษะ', 'ระวัง', 'เว็บไซต์', 'หลอกลวง']

-- tokenizer ใหม่ (attacut) --
['นายก', 'รัฐมนตรี', 'เรียนรู้', 'วิธี', 'ดี', 'พัฒนา', 'ทักษะ', 'ระวัง', 'เว็บไซต์', 'หลอกลวง']

🔸 ตัวอย่างที่ 2:
ข้อความ: ปชช.โวย! ราคาหมูแพงทะลุ 300 บาท/กก. #Saveหมูแพง @PriceControl

-- tokenizer เก่า --
['ปชช', 'โวย', 'ราคา', 'หมู', 'แพง', 'ทะลุ', '300', 'บาท', 'กก', 'แพง']

-- tokenizer ใหม่ (auto) --
['ปชช', 'โวย', 'ราคา', 'หมู', 'แพง', 'ทะลุ', 'บาท', 'กก', 'แพง']

-- tokenizer ใหม่ (attacut) --
['นายก', 'รัฐมนตรี', 'เรียนรู้', 'วิธี', 'ดี', 'พัฒนา', 'ทักษะ', 'ระวัง', 'เว็บไซต์', 'หลอกลวง']

-

## 🔄 อัปเดต Pipeline

หลังจากทดสอบ tokenizer ใหม่แล้ว ถ้าต้องการใช้ tokenizer ใหม่กับข้อมูลทั้งหมด ให้รันเซลล์ด้านล่างเพื่ออัปเดต pipeline:

1. อัปเดตคอลัมน์ `Tokens` และ `TokenStr` ด้วย tokenizer ใหม่
2. สร้าง TF-IDF matrix ใหม่
3. บันทึกไฟล์ผลลัพธ์

⚠️ หมายเหตุ: 
- แนะนำให้ติดตั้ง attacut ก่อน (`pip install attacut`) เพื่อให้ได้ผลลัพธ์ที่ดีที่สุด
- การรันใหม่จะใช้เวลาสักครู่เนื่องจากต้อง tokenize ข้อมูลทั้งหมดอีกครั้ง

In [11]:
pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   --------------------------------------- 914.9/914.9 kB 14.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 24.8 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\SUPHASET\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm  # ใช้ tqdm ธรรมดาแทน tqdm.notebook
import pandas as pd

# อัปเดต pipeline ด้วย tokenizer ใหม่
print("🔄 เริ่มอัปเดต pipeline...")

# 1. อัปเดต Tokens column
print("\n1️⃣ Tokenizing ข้อมูลใหม่...")
X_under["Text"] = X_under["Title"].fillna("") + " " + X_under["Body"].fillna("")

# ใช้ list comprehension with tqdm แทน progress_apply
print("กำลัง tokenize ข้อความ...")
tokens = [hybrid_tokenizer_improved(text, tokenizer_preference='attacut') 
         for text in tqdm(X_under["Text"], desc="Tokenizing")]
X_under["Tokens"] = tokens
X_under["TokenStr"] = X_under["Tokens"].apply(lambda x: " ".join(x))

# 2. สร้าง TF-IDF matrix ใหม่
print("\n2️⃣ สร้าง TF-IDF matrix...")
vectorizer = TfidfVectorizer(
    analyzer="word",
    max_df=0.9,
    min_df=1,
    stop_words=list(all_stopwords),
    ngram_range=(1, 2),
    max_features=10000
)

X_tfidf = vectorizer.fit_transform(X_under["TokenStr"])

# แสดงผล
print("\n✅ สร้าง TF-IDF สำเร็จ!")
print("Shape:", X_tfidf.shape)
print("\n🔸 ตัวอย่าง 20 คำแรกใน vocabulary:")
print(vectorizer.get_feature_names_out()[:20])

print("\n🔸 ตัวอย่าง Tokens จาก 3 ข่าวแรก:")
print(X_under[["Text", "Tokens"]].head(3))

🔄 เริ่มอัปเดต pipeline...

1️⃣ Tokenizing ข้อมูลใหม่...
กำลัง tokenize ข้อความ...
กำลัง tokenize ข้อความ...


Tokenizing:   0%|          | 9/4245 [00:06<38:36,  1.83it/s]  

In [5]:
# บันทึกผลลัพธ์
print("3️⃣ บันทึกผลลัพธ์...")

# บันทึก TF-IDF vectorizer
joblib.dump(vectorizer, 'result/vectorizer.pkl')

# บันทึก X_tfidf (sparse matrix)
joblib.dump(X_tfidf, 'result/X_tfidf.pkl')

print("✅ บันทึกข้อมูลเรียบร้อยแล้ว!")
print("\nℹ️ หมายเหตุ: ถ้าต้องการใช้ผลลัพธ์ใหม่ ให้รัน notebook อื่น ๆ ใหม่ด้วย (NeuralNetwork.ipynb)")

3️⃣ บันทึกผลลัพธ์...
✅ บันทึกข้อมูลเรียบร้อยแล้ว!

ℹ️ หมายเหตุ: ถ้าต้องการใช้ผลลัพธ์ใหม่ ให้รัน notebook อื่น ๆ ใหม่ด้วย (NeuralNetwork.ipynb)
